In [ ]:
# Euchre Calculator -- one hand from deal to score, using the API as it stands.
import collections
import random
import time

import numpy as np

import bidding as b
import game
import rotation as r
from fast_search import definitive_winner

In [ ]:
# Deal. Four hands, the up-card turned off the kitty, three buried under it,
# and a dealer. No trump yet -- nobody has called.
#
# The dealer bids LAST, so with seat 3 dealing the eldest hand is seat 0.
deal = game.deal_random(rng=random.Random(7), dealer=3)

print(deal.describe())
print("\nbidding order:", deal.bidding_order(), "(eldest first, dealer last)")

  seat 0: AC 9S KS QH TD
  seat 1: JH AS QS 9D TS
  seat 2: TH KH JD 9C QD
  seat 3 (dealer): AD QC AH JC TC
  up-card: JS
  buried:  9H KC KD

bidding order: [0, 1, 2, 3] (eldest first, dealer last)


In [ ]:
# Order or pass, seat by seat, then the score.
#
# outcome.value is net points to team 0 (seats 0 and 2), so calls by different
# seats are comparable. contract.solve() gives the same hand from the CALLING
# team's side, which is the sign convention definitive_winner uses.
outcome = b.solve_bidding(deal)

for step in outcome.line:
    print(" ", step)

contract = outcome.contract
print("\ncontract      :", contract)
print("trump         :", r.suit_name(contract.trump))
print("caller        : seat %d (team %d)" % (contract.caller, contract.caller_team))
print("dealer pitched:", r.card_name(contract.discard))
print("\nscore to the calling team:", contract.solve())
print("net to team 0            : %+d" % outcome.value)

  seat 0 passes
  seat 1 passes
  seat 2 passes
  seat 3 orders up spades

contract      : seat 3 ordered up spades (dealer pitched AD)
trump         : spades
caller        : seat 3 (team 1)
dealer pitched: AD

score to the calling team: 2
net to team 0            : -2


In [ ]:
# The hands the contract is actually played from, and the play itself.
#
# deal_to_engine rotates the called suit into the solver's canonical frame;
# definitive_winner prints the line in that frame, so cards come out as the
# 2-D vectors rather than names.
hands = r.deal_to_engine(contract.deal.hands, contract.trump)

definitive_winner(dealt_hands=hands,
                  starting_player=contract.deal.first_bidder,
                  caller=contract.caller,
                  verbose=True)

Starting hands:
 [[[  0 -14]
  [  0  90]
  [  0 120]
  [-12   0]
  [ 10   0]]

 [[-11   0]
  [  0 130]
  [  0 110]
  [  9   0]
  [  0 100]]

 [[-10   0]
  [-13   0]
  [ 11   0]
  [  0  -9]
  [ 12   0]]

 [[  0 -12]
  [-14   0]
  [  0 135]
  [  0 -10]
  [  0 140]]]
Trick 1: [[0, -14], [0, 130], [0, -9], [0, -12]]  (played by [0, 1, 2, 3])
Trick 1 winner: 1
Trick 2: [[-11, 0], [-10, 0], [-14, 0], [-12, 0]]  (played by [1, 2, 3, 0])
Trick 2 winner: 3
Trick 3: [[0, 140], [0, 90], [0, 100], [12, 0]]  (played by [3, 0, 1, 2])
Trick 3 winner: 3
Trick 4: [[0, 135], [0, 120], [0, 110], [11, 0]]  (played by [3, 0, 1, 2])
Trick 4 winner: 3
Trick 5: [[0, -10], [10, 0], [9, 0], [-13, 0]]  (played by [3, 0, 1, 2])
Trick 5 winner: 3
Final result: [1, 3, 3, 3, 3]


In [ ]:
# "Should I order this up?" -- pin the hand you can see, deal the rest.
#
# first_bid_choice returns the two options from the eldest hand's OWN side, so
# the bigger number is the better bid. Here passing wins.
my_seat = 0
my_hand = r.parse_hand("JS AS 9H 9D TC")      # right bower, ace of trump, junk
up_card = r.parse_card("9S")

one = game.deal_around(known_hand=my_hand, seat=my_seat, up_card=up_card,
                       rng=random.Random(1), dealer=3)

ordered, passed = b.first_bid_choice(one)
print("holding %s, up-card %s" % (r.hand_name(my_hand), r.card_name(up_card)))
print("this layout -> order %+d / pass %+d" % (ordered, passed))

holding JS AS 9H 9D TC, up-card 9S
this layout -> order -2 / pass -1


In [ ]:
# One layout tells you nothing -- the other nineteen cards fell a particular
# way. Re-deal them a few hundred times and solve the whole auction each time.
#
# A full auction is ~36 trick-play solves, so roughly 13 ms per deal. Sampling
# error is the only error here, since each solve is exact.
N = 400
rng = random.Random(2024)

b.solve_bidding(game.deal_around(known_hand=my_hand, seat=my_seat,
                                 up_card=up_card, rng=random.Random(0),
                                 dealer=3))          # warm the JIT

values, callers, trumps = [], [], []
t0 = time.perf_counter()
for _ in range(N):
    d = game.deal_around(known_hand=my_hand, seat=my_seat, up_card=up_card,
                         rng=rng, dealer=3)
    o = b.solve_bidding(d)
    values.append(b.value_to(my_seat, o.value))      # net points to MY team
    callers.append(None if o.passed_out else o.contract.caller)
    trumps.append(None if o.passed_out else o.contract.trump)
elapsed = time.perf_counter() - t0

values = np.array(values)
se = values.std(ddof=1) / np.sqrt(N)

print("holding %s as seat %d (eldest), up-card %s"
      % (r.hand_name(my_hand), my_seat, r.card_name(up_card)))
print("%d auctions in %.1fs (%.0f ms each)\n" % (N, elapsed, 1000 * elapsed / N))
print("expected net points to my team: %+.3f   95%% CI [%+.3f, %+.3f]"
      % (values.mean(), values.mean() - 1.96 * se, values.mean() + 1.96 * se))

print("\noutcome distribution:")
for v, n in sorted(collections.Counter(values.tolist()).items()):
    print("  %+d : %5.1f%%" % (v, 100 * n / N))

print("\nwho ends up calling:")
for seat, n in sorted(collections.Counter(callers).items(), key=lambda kv: -kv[1]):
    label = ("passed out" if seat is None else
             "seat %d (%s)" % (seat, "my team" if seat % 2 == my_seat % 2
                               else "opponents"))
    print("  %-22s %5.1f%%" % (label, 100 * n / N))

print("\ntrump called:")
for suit, n in sorted(collections.Counter(trumps).items(), key=lambda kv: -kv[1]):
    print("  %-10s %5.1f%%" % ("none" if suit is None else r.suit_name(suit),
                               100 * n / N))

# Every seat above bids and plays with all four hands visible. That is exact,
# and it is not what happens at a table. Solved this way the auction almost
# never passes out (0 of 1600 measured), because somebody can nearly always
# find a call that is at worst harmless -- real tables throw hands in
# constantly. Treat it as the baseline, not the answer.
#
# Not yet modelled: loners, and conditioning the unseen cards on the bidding.

holding JS AS 9H 9D TC as seat 0 (eldest), up-card 9S
400 auctions in 5.0s (13 ms each)

expected net points to my team: +0.400   95% CI [+0.274, +0.526]

outcome distribution:
  -2 :   4.8%
  -1 :  34.0%
  +1 :  39.0%
  +2 :  22.2%

who ends up calling:
  seat 0 (my team)        43.2%
  seat 3 (opponents)      38.8%
  seat 2 (my team)        18.0%

trump called:
  clubs       41.2%
  spades      40.2%
  diamonds    11.2%
  hearts       7.2%


In [ ]:
# Narrower question, ~6x cheaper: I AM ordering -- what do I score?
#
# order_up() fixes that my_seat calls the turned suit and minimaxes only the
# dealer's discard, over the five cards it holds plus the up-card it just took.
# That is 6 trick-play solves rather than the 32 a whole auction needs.
#
# Not the same question as the cell above. There I might pass and let someone
# else call, so that EV includes hands I never played. Here I always call.
N = 10_000
rng = random.Random(2024)

b.order_up(game.deal_around(known_hand=my_hand, seat=my_seat, up_card=up_card,
                            rng=random.Random(0), dealer=3), my_seat)   # warm

ordered, pitched = [], []
t0 = time.perf_counter()
for _ in range(N):
    d = game.deal_around(known_hand=my_hand, seat=my_seat, up_card=up_card,
                         rng=rng, dealer=3)
    value, contract = b.order_up(d, my_seat)
    ordered.append(b.value_to(my_seat, value))          # net points to MY team
    pitched.append(contract.discard.suit == up_card.suit)
elapsed = time.perf_counter() - t0

ordered = np.array(ordered)
se = ordered.std(ddof=1) / np.sqrt(N)

print("ordering up %s, holding %s as seat %d"
      % (r.suit_name(up_card.suit), r.hand_name(my_hand), my_seat))
print("%d deals in %.0fs (%.1f ms each)\n" % (N, elapsed, 1000 * elapsed / N))
print("EV if I order: %+.3f   95%% CI [%+.3f, %+.3f]"
      % (ordered.mean(), ordered.mean() - 1.96 * se, ordered.mean() + 1.96 * se))

print("\noutcome distribution:")
for v, n in sorted(collections.Counter(ordered.tolist()).items()):
    print("  %+d : %5.1f%%" % (v, 100 * n / N))

print("\neuchred %.1f%% of the time; the dealer pitched a trump on %.1f%%"
      % (100 * (ordered == -2).mean(), 100 * np.mean(pitched)))

# Worth sitting next to the cell above: ordering here is worth +0.016, while
# letting the auction run is worth +0.400 on the same hand. This hand is not
# one to order -- the difference is the hands where passing lets a better call
# happen, mine or an opponent's I can beat.

ordering up spades, holding JS AS 9H 9D TC as seat 0
10000 deals in 21s (2.1 ms each)

EV if I order: +0.016   95% CI [-0.015, +0.048]

outcome distribution:
  -2 :  37.6%
  +1 :  48.0%
  +2 :  14.4%

euchred 37.6% of the time; the dealer pitched a trump on 12.5%
